# Lab 00: stima della memoria e anatomia di un GGUF

Lab: `corso/laboratori/lab-00-01-stima-memoria e lab-00-02-anatomia-gguf`

Esegui le celle in ordine (Runtime > Esegui tutto). Scarica fino a 1 GB e richiede alcuni minuti.


## 1. Preparazione

Scarica i binari ufficiali di llama.cpp (build `b11046`, quella del corso) e il materiale del lab.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # thread = core disponibili (oltre i core fisici si rallenta)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Scarica il modello di prova (785 MiB, licenza Apache 2.0) nella cache di Hugging Face.

In [ ]:
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF 2>&1 | tail -3
!llama cli -cl

## 2. Stima della memoria (lab 00-01)

Lo script `stima_memoria.py` applica le formule del modulo 00: pesi + KV cache + overhead.

In [ ]:
%%bash
cd $LABS/lab-00-01-stima-memoria
python3 stima_memoria.py --preset qwen3-8b --quant q4_k_m --ctx 8192
python3 stima_memoria.py --params 7.6e9 --quant q4_k_m --ctx 8192 --layers 28 --kv-heads 4 --head-dim 128 | tail -8

**Prova tu:** cambia `--ctx` in 32768 e osserva quale voce cresce.

## 3. Anatomia di un file GGUF (lab 00-02)

In [ ]:
!pip -q install gguf

In [ ]:
%%bash
M=$(ls ~/.cache/huggingface/hub/models--ggml-org--Qwen3.5-0.8B-GGUF/snapshots/*/*.gguf)
ls -la "$M" "$(readlink -f "$M")"
gguf-dump --no-tensors "$M" | head -24
cd $LABS/lab-00-02-anatomia-gguf
python3 anatomia_gguf.py "$M"